# Analisis Eksperimen B1 — Structured Prompted LLM (Few-Shot + Abstain Policy)

- **Konfigurasi:** `B1` (Base model sama dengan B0, ditambah prompt terstruktur, skema JSON, *few-shot* train, dan kebijakan *abstain* — **masih tanpa retrieval**)
- **Base Model:** `Qwen/Qwen2.5-1.5B-Instruct` | **Seed:** `42` | **Split:** `dev.jsonl` (30 query)
- **Pertanyaan Riset (`NLP_IR.pdf`):** Seberapa besar efek rekayasa prompt murni (*structured prompting + abstain policy*) terhadap B0, dan error apa yang tetap tidak bisa diselesaikan tanpa retrieval?

In [ ]:
import json, statistics
from pathlib import Path

ROOT = Path('..') if (Path('..') / 'runs').exists() else Path('.')
s_b0 = json.loads((ROOT / 'runs/B0/summary.json').read_text(encoding='utf-8'))
s_b1 = json.loads((ROOT / 'runs/B1/summary.json').read_text(encoding='utf-8'))
rows = [json.loads(l) for l in (ROOT / 'runs/B1/dev_outputs.jsonl').open(encoding='utf-8') if l.strip()]

print('=== PERBANDINGAN DELTA B0 -> B1 ===')
for k in ['json_valid', 'abstain_acc_unans', 'false_refusal_ans', 'citation_rate_ans', 'avg_latency_s']:
    delta = s_b1[k] - s_b0[k]
    print(f'{k:20s}: B0={s_b0[k]:6.3f} -> B1={s_b1[k]:6.3f} (delta={delta:+.3f})')

In [ ]:
# Breakdown per Kategori + Distribusi Verdict
by_cat = {}
for r in rows:
    by_cat.setdefault(r['category'], []).append(r)

print(f"{'Kategori':18s} | {'N':>2s} | {'JSON':>4s} | {'Sup':>3s} | {'Unk':>3s} | {'Cit':>3s} | {'AvgOut':>6s} | {'AvgLat':>6s}")
print('-' * 72)
for cat, items in by_cat.items():
    jv = sum(1 for x in items if x['parsed_json'] is not None)
    sup = sum(1 for x in items if (x['parsed_json'] or {}).get('verdict') == 'supported')
    unk = sum(1 for x in items if (x['parsed_json'] or {}).get('verdict') == 'unknown')
    cit = sum(1 for x in items if len((x['parsed_json'] or {}).get('citations') or []) > 0)
    ao = sum(x['out_tokens'] for x in items) / len(items)
    al = sum(x['latency_s'] for x in items) / len(items)
    print(f'{cat:18s} | {len(items):2d} | {jv:4d} | {sup:3d} | {unk:3d} | {cit:3d} | {ao:6.1f} | {al:5.2f}s')

lats = sorted(r['latency_s'] for r in rows)
print(f'\nB1 Latency -> mean={statistics.mean(lats):.2f}s | p50={statistics.median(lats):.2f}s | p95={lats[int(0.95*len(lats))]:.2f}s')

In [ ]:
# Audit Error 1: Mengapa 3 Query Gagal JSON Parsing di B1?
bad = [r for r in rows if r['parsed_json'] is None]
print(f'Jumlah JSON gagal parse di B1: {len(bad)}/{len(rows)}')
for r in bad:
    print('=' * 80)
    print(f"{r['query_id']} [{r['category']}] out_tok={r['out_tokens']}")
    print('Q:', r['query'])
    print('RAW:', r['raw_output'])

In [ ]:
# Audit Error 2: Abstain Behavior pada Unanswerable (6 query) & False Refusal pada Answerable
print('--- UNANSWERABLE (harus unknown) ---')
for r in rows:
    if r['answerability'] == 'unanswerable':
        pj = r['parsed_json'] or {}
        print(f"{r['query_id']} | verdict={pj.get('verdict')} | Q: {r['query'][:75]}")
        print('   answer:', str(pj.get('answer', r['raw_output']))[:160])

print('\n--- FALSE REFUSAL (answerable tapi verdict=unknown) ---')
for r in rows:
    if r['answerability'] == 'answerable' and (r['parsed_json'] or {}).get('verdict') == 'unknown':
        pj = r['parsed_json']
        print(f"{r['query_id']} [{r['category']}] | Q: {r['query']}")
        print('   abstain_reason:', pj.get('abstain_reason'))

In [ ]:
# Audit Error 3: Confident Hallucination di Balik JSON Valid (Contoh Pasal KHI)
for r in rows:
    if r['category'] == 'pasal_definisi':
        pj = r['parsed_json'] or {}
        print(f"{r['query_id']} | cit={pj.get('citations')} | Q: {r['query']}")
        print('   A:', str(pj.get('answer', ''))[:240])
        print()

### Kesimpulan Analisis B1 (Structured Prompted LLM)
1. **Lompatan Kepatuhan Format & Efisiensi vs B0:** `json_valid` melonjak dari **0% → 90% (27/30)**, dan rata-rata latensi justru turun drastis dari **14,34 detik → 8,23 detik** (0 output terpotong) karena skema JSON memaksa jawaban ringkas.
2. **Efek Kebijakan Abstain Mulai Bekerja:** `abstain_acc_unans` naik dari **0% → 33,3% (2/6)** (`Q_082` adat Batak Toba dan `Q_084` permintaan PII berhasil ditolak), dengan *false refusal* **12,5% (3/24)** pada query *answerable*.
3. **Kelemahan Fatal Tanpa Retrieval — *Confident Hallucination*:** Meskipun `citation_rate_ans` terlihat tinggi (**83,3%**), sitasi dan isi hukumnya banyak yang **dikarang sendiri** karena tidak ada konteks dokumen. Contoh nyata pada `Q_016`: model menulis JSON rapi berisi *"Pasal 182 KUH Perdata... yaitu 3/5 dari jumlah saudara kandung"* (keliru rezim hukum dan keliru angka).
4. **Implikasi Riset:** Prompting saja mampu memperbaiki **format dan kepatuhan struktur**, tetapi **tidak mampu menyembuhkan halusinasi sumber hukum** — ini menjadi dasar kuat mengapa kita perlu menambahkan retrieval di **B2**.